# Time Correction Visualization Pipeline

This notebook provides a step-by-step interactive visualization for evaluating **Time Correction** results.
It compares **Raw** data (Before Time Correction) with **Time-Corrected** data (After Time Correction).

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
import os

# Add the project root to sys.path so we can import CyTof_BatchCorrection
project_root = os.path.dirname(os.getcwd())
if project_root not in sys.path:
    sys.path.append(project_root)
print(f"Project root added to sys.path: {project_root}")


#### This is for the visualization of the Time-corrected results

In [ ]:
import json
import os
import pickle
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from glob import glob
from collections import defaultdict
import collections.abc
collections.MutableMapping = collections.abc.MutableMapping
import scanpy as sc
import seaborn as sns
import anndata as ad
from matplotlib.lines import Line2D
%matplotlib inline
from src.utils import *
from src.time_correction_visualization import *
from src.batch_correction_visualization import *

#### Main pipline for the Time Correction Visualization

In [ ]:
# ---- USER INPUT -----------------------------------------------------------
study_config_path = "../results/study_config.json"
# For large datasets such as D24, keep BOTH set to one dataset/panel.
# If D24 was prepared as split configs, point study_config_path to the target panel config.
target_dataset_override = None  # e.g. "D0", "D24", or None
target_panel_override   = "Panel1" # e.g. "Panel1" or "Panel2"

# Chunked reading prevents loading a merged matrix before filtering.
matrix_chunksize = 100_000
# Decide from the original filtered row count whether to load all cells or sample.
matrix_sampling_threshold = 10_000_000
# Above the threshold, keep at most N cells per sample_name + plate.
panel_sample_per_group = 20000
panel_sample_group_cols = ("sample_name", "plate")
# --------------------------------------------------------------------------

def read_matrix_filtered_grouped(
    path,
    dataset=None,
    panel=None,
    chunksize=250_000,
    sample_per_group=None,
    group_cols=("sample_name", "plate"),
    random_state=42,
    low_memory=False,
):
    """
    Read a matrix CSV with optional dataset/panel filtering and balanced streaming sampling.

    Unlike read_matrix_filtered(..., sample_per_sample=...), this samples per group
    such as sample_name + plate, so repeated anchor/sample names are preserved across plates.
    """
    if sample_per_group is None or sample_per_group <= 0:
        return read_matrix_filtered(
            path,
            dataset=dataset,
            panel=panel,
            chunksize=chunksize,
            sample_per_sample=None,
            random_state=random_state,
            low_memory=low_memory,
        )

    if chunksize is None or chunksize <= 0:
        df = read_matrix_filtered(
            path,
            dataset=dataset,
            panel=panel,
            chunksize=None,
            sample_per_sample=None,
            random_state=random_state,
            low_memory=low_memory,
        )
        present_group_cols = [c for c in group_cols if c in df.columns]
        if not present_group_cols:
            return df.sample(n=min(len(df), sample_per_group), random_state=random_state).reset_index(drop=True)
        return (
            df.groupby(present_group_cols, observed=True, group_keys=False)
              .apply(lambda g: g.sample(n=min(len(g), sample_per_group), random_state=random_state))
              .reset_index(drop=True)
        )

    kept = []
    counts = {}
    rng_seed = int(random_state)

    for chunk_idx, chunk in enumerate(pd.read_csv(path, chunksize=chunksize, low_memory=low_memory)):
        if dataset is not None and "dataset" in chunk.columns:
            chunk = chunk[chunk["dataset"].astype(str) == str(dataset)]
        if panel is not None and "panel" in chunk.columns:
            chunk = chunk[chunk["panel"].astype(str) == str(panel)]
        if chunk.empty:
            continue

        present_group_cols = [c for c in group_cols if c in chunk.columns]
        if not present_group_cols:
            key = ("__all__",)
            remaining = sample_per_group - counts.get(key, 0)
            if remaining <= 0:
                continue
            take = min(len(chunk), remaining)
            kept.append(chunk.sample(n=take, random_state=rng_seed + chunk_idx))
            counts[key] = counts.get(key, 0) + take
            continue

        sampled_groups = []
        for group_key, group in chunk.groupby(present_group_cols, observed=True):
            if not isinstance(group_key, tuple):
                group_key = (group_key,)
            key = tuple(str(x) for x in group_key)
            remaining = sample_per_group - counts.get(key, 0)
            if remaining <= 0:
                continue
            take = min(len(group), remaining)
            sampled_groups.append(group.sample(n=take, random_state=rng_seed + chunk_idx))
            counts[key] = counts.get(key, 0) + take
        if sampled_groups:
            kept.append(pd.concat(sampled_groups, ignore_index=True))

    if not kept:
        return pd.DataFrame()
    return pd.concat(kept, ignore_index=True)


def count_matrix_rows_filtered(path, dataset=None, panel=None, chunksize=250_000):
    """Count matching CSV rows without materializing marker columns."""
    columns = list(pd.read_csv(path, nrows=0).columns)
    filter_cols = [c for c in ("dataset", "panel") if c in columns]
    usecols = filter_cols or [columns[0]]
    total = 0
    for chunk in pd.read_csv(path, usecols=usecols, chunksize=chunksize):
        keep = np.ones(len(chunk), dtype=bool)
        if dataset is not None and "dataset" in chunk.columns:
            keep &= chunk["dataset"].astype(str).eq(str(dataset)).to_numpy()
        if panel is not None and "panel" in chunk.columns:
            keep &= chunk["panel"].astype(str).eq(str(panel)).to_numpy()
        total += int(keep.sum())
    return total


def resolve_matrix_sampling(path, dataset, panel):
    total = count_matrix_rows_filtered(
        path, dataset=dataset, panel=panel, chunksize=matrix_chunksize
    )
    use_sampling = total > matrix_sampling_threshold
    sample_cap = panel_sample_per_group if use_sampling else None
    mode = f"sample (max {sample_cap:,}/sample_name+plate)" if use_sampling else "full data"
    print(
        f"[LOAD] Source rows for {dataset}/{panel}: {total:,} | "
        f"threshold={matrix_sampling_threshold:,} -> {mode}"
    )
    return sample_cap, total


def normalize_group_labels(df):
    """Normalize biological group labels to the plotting convention used in prior figures."""
    if df is None or df.empty or "group" not in df.columns:
        return df
    df = df.copy()
    group = df["group"].astype("string").str.strip()
    group_lower = group.str.lower()
    sample = df["sample_name"].astype("string") if "sample_name" in df.columns else pd.Series("", index=df.index, dtype="string")
    sample_lower = sample.str.lower()

    healthy_mask = group_lower.isin(["healthy", "hbm", "normal", "control"])
    healthy_mask = healthy_mask | (group_lower.isin(["unknown", "nan", "none", ""]) & sample_lower.str.contains("healthy|hbm", na=False))
    df.loc[healthy_mask, "group"] = "healthy"
    df.loc[group_lower.eq("adult"), "group"] = "adult"
    df.loc[group_lower.eq("pediatric"), "group"] = "pediatric"
    return df

with open(study_config_path) as f:
    study_config = json.load(f)

config = get_config(config_dict=study_config["cytof_config"])
ctx    = CyTOFContext(config)

target_dataset = target_dataset_override or study_config["dataset"]
target_panel   = target_panel_override or study_config.get("panel", "Panel1")
anchor_name    = ctx.anchors.get(target_dataset)

with open(study_config["outputs"]["markers_path"]) as f:
    markers = json.load(f)

raw_path          = study_config["outputs"]["matrix_path"]
intermediate_path = study_config["outputs"]["matrix_intermediate_path"]

effective_sample_per_group, source_cell_count = resolve_matrix_sampling(
    raw_path, target_dataset, target_panel
)

print(f"[LOAD] Reading only dataset={target_dataset}, panel={target_panel}")
df_raw = read_matrix_filtered_grouped(
    raw_path,
    dataset=target_dataset,
    panel=target_panel,
    chunksize=matrix_chunksize,
    sample_per_group=effective_sample_per_group,
    group_cols=panel_sample_group_cols,
)
df_a = read_matrix_filtered_grouped(
    intermediate_path,
    dataset=target_dataset,
    panel=target_panel,
    chunksize=matrix_chunksize,
    sample_per_group=effective_sample_per_group,
    group_cols=panel_sample_group_cols,
)

# Apply arcsinh to raw data
numeric_markers = [m for m in markers if m in df_raw.columns]
df_b = arcsinh(df_raw, numeric_markers)
df_b = normalize_group_labels(df_b)
df_a = normalize_group_labels(df_a)
del df_raw

print(f"Dataset  : {target_dataset}")
print(f"Panel    : {target_panel}")
print(f"Anchor   : {anchor_name}")
print(f"Before (arcsinh raw)            : {len(df_b)} cells")
print(f"After  (arcsinh time-corrected) : {len(df_a)} cells")
print(f"Markers ({len(markers)}): {markers[:5]}...")


In [ ]:
# --- GENERATE & LOAD STUDY-SPECIFIC COLORS ---
_results_dir     = os.path.dirname(study_config["outputs"]["matrix_corrected_path"])
_study_name      = study_config.get("study_name", "study")
colors_save_path = os.path.join(_results_dir, f"colors_dict_{_study_name}.pkl")

if df_a is not None and not os.path.exists(colors_save_path):
    print("\n===== GENERATING GLOBAL COLORS FROM GATED DATA =====")
    all_cell_types = set()
    tc_dir = os.path.join(ctx.output_dir, "time_corrected")
    if os.path.exists(tc_dir):
        for f in glob(os.path.join(tc_dir, "*.csv")):
            try:
                df_temp = pd.read_csv(f, usecols=["cell_type"])
                all_cell_types.update(df_temp["cell_type"].dropna().unique())
            except: pass
    cols = ["celltypes", "panel", "plate", "sample_name", "dataset"]
    temp_df = df_a[cols].drop_duplicates().copy()
    for col in cols:
        if col not in temp_df.columns:
            temp_df[col] = np.nan
    temp_df["cell_type"] = pd.Series(list(all_cell_types))
    generate_and_save_colors(df=temp_df, save_path=colors_save_path)
elif os.path.exists(colors_save_path):
    print(f"[INFO] Using existing colors from {colors_save_path}")

global_colors = {}
if os.path.exists(colors_save_path):
    with open(colors_save_path, "rb") as f_in:
        global_colors = pickle.load(f_in)
    print(f"[INFO] Global colors loaded ({len(global_colors)} keys)")

In [ ]:
import gc

df_b = normalize_group_labels(df_b)
df_a = normalize_group_labels(df_a)

factors = [c for c in ["plate", "sample_name", "celltypes", "Time"] if c in df_b.columns]

# Loading already chose full data versus grouped sampling from the source row count.
analysis_subsample = None
umap_subsample = 50_000

def _paired_sample(before, after, n, random_state=42):
    if n is None or max(len(before), len(after)) <= n:
        return before, after
    return (
        before.sample(n=min(len(before), n), random_state=random_state).copy(),
        after.sample(n=min(len(after), n), random_state=random_state).copy(),
    )

# df_b and df_a are already filtered to target_panel — no loop needed
panel_markers = [
    m for m in markers
    if m in df_b.columns
    and m in ctx.true_markers
    and df_b[m].notna().any()
]
print(f"Panel    : {target_panel}  |  Dataset: {target_dataset}")
print(f"Markers  : {len(panel_markers)} | Before : {len(df_b)} | After : {len(df_a)}")

# Per-panel colors
current_colors = {}
if global_colors and os.path.exists(colors_save_path):
    current_colors = load_and_subset_colors(df_a, colors_save_path)

# ── 1. PCA Variance Analysis ──────────────────────────────────────────
print(f"\n[ANALYZE] Running variation_pca for {target_panel}...")
var_b = variation_pca(df_b, markers=panel_markers, group_col=None,
                      info_cols=factors, panel=target_panel)
var_a = variation_pca(df_a, markers=panel_markers, group_col=None,
                      info_cols=factors, panel=target_panel)

print("\nVariance Results (Before Time Correction):")
print(var_b)
print("\nVariance Results (After Time Correction):")
print(var_a)

# ── 2. UMAP ───────────────────────────────────────────────────────────
print(f"\n[ANALYZE] Generating UMAP for {target_panel}...")
try:
    _umap_b, _umap_a = _paired_sample(df_b, df_a, umap_subsample, random_state=42)
    print(f"  [INFO] UMAP sample: before={len(_umap_b)}, after={len(_umap_a)}")
    adata = umap_by_celltype(_umap_b, _umap_a, panel=target_panel,
                             markers=panel_markers, celltype_col="celltypes",
                             subsample=None)
    del _umap_b, _umap_a
    gc.collect()
    plt.show()
except Exception as _e:
    print(f"  [WARN] UMAP skipped: {_e}")

# ── 3. Signal Distributions ───────────────────────────────────────────
print(f"\n[ANALYZE] Running signal distribution plots for {target_panel}...")
_plot_b, _plot_a = _paired_sample(df_b, df_a, analysis_subsample, random_state=43)
print(f"  [INFO] Signal distribution sample: before={len(_plot_b)}, after={len(_plot_a)}")
plot_signal_distributions_joint(df_before=_plot_b, df_after=_plot_a,
                                panel=target_panel, markers=panel_markers,
                                save_dir=None, subsample=50000)
del _plot_b, _plot_a
gc.collect()
plt.show()

# ── 4. KL Divergence + MWU ────────────────────────────────────────────
print(f"\n[ANALYZE] Calculating KL divergence and MWU for {target_panel}...")
_stat_b, _stat_a = _paired_sample(df_b, df_a, analysis_subsample, random_state=44)
print(f"  [INFO] KD/MWU sample: before={len(_stat_b)}, after={len(_stat_a)}")
kd_df  = calculate_kd(df_before=_stat_b, df_after=_stat_a, panel=target_panel, ctx=ctx)
mwu_df = calculate_mwu(df_before=_stat_b, df_after=_stat_a, panel=target_panel, ctx=ctx)
del _stat_b, _stat_a
gc.collect()
plot_kd_distribution(kd_df, save_path=None)
plt.show()
plot_kd_with_pvalue(kd_df, mwu_df, save_path=None)
plt.show()

# ── 5. Marker-level Variance Boxplot ──────────────────────────────────
print(f"\n[ANALYZE] Generating Marker-level variance boxplots for {target_panel}...")
base_panel_dir = os.path.join(ctx.output_dir, target_dataset, target_panel)
path_Variance  = os.path.join(base_panel_dir, "Time_Variance")
os.makedirs(path_Variance, exist_ok=True)

_mvar_b_df, _mvar_a_df = _paired_sample(df_b, df_a, analysis_subsample, random_state=45)
print(f"  [INFO] Marker variance sample: before={len(_mvar_b_df)}, after={len(_mvar_a_df)}")
mvar_b = variation_marker(_mvar_b_df, markers=panel_markers, group_col=None,
                          info_cols=factors, panel=target_panel)
mvar_a = variation_marker(_mvar_a_df, markers=panel_markers, group_col=None,
                          info_cols=factors, panel=target_panel)
del _mvar_b_df, _mvar_a_df
gc.collect()

mvar_b.to_csv(os.path.join(path_Variance,
    f"variance_marker_raw_before_{target_dataset}_{target_panel}_Time.csv"), index=False)
mvar_a.to_csv(os.path.join(path_Variance,
    f"variance_marker_raw_after_{target_dataset}_{target_panel}_Time.csv"), index=False)

plot_variance_marker_boxplot(
    var_marker_raw=mvar_b,
    var_marker_corr=mvar_a,
    save_path=os.path.join(path_Variance,
        f"variance_marker_boxplot_{target_dataset}_{target_panel}_Time.png"),
    use_log=True
)
plt.show()

try:
    del adata
except NameError:
    pass
gc.collect()
print(f"\n[DONE] Finished panel: {target_panel}")


In [ ]:
import gc
from IPython.display import Image, display

# Group-stratified variance analysis.
# This intentionally recalculates variance with group_col="group" and is
# different from the preceding cell's overall analysis (group_col=None).
gc.collect()


def _configured_panels(cfg, current_panel):
    """Return panels explicitly declared by the current study config."""
    configured = cfg.get("panels")
    if isinstance(configured, dict):
        panels = list(configured)
    elif isinstance(configured, (list, tuple)):
        panels = list(configured)
    else:
        panels = [cfg.get("panel", current_panel)]

    panels = [str(p) for p in panels if p]
    if str(current_panel) not in panels:
        panels.insert(0, str(current_panel))
    return list(dict.fromkeys(panels))


def _grouped_panel_variance(df_before, df_after, panel_name, panel_markers):
    """Recalculate PCA/ANOVA variance separately for each biological group."""
    df_before = df_before.copy()
    df_after = df_after.copy()

    # datasets use `group`. Create one canonical grouping column.
    if "group" not in df_before.columns and "condition" in df_before.columns:
        df_before["group"] = df_before["condition"]
    if "group" not in df_after.columns and "condition" in df_after.columns:
        df_after["group"] = df_after["condition"]

    df_before = normalize_group_labels(df_before)
    df_after = normalize_group_labels(df_after)

    valid_markers = [
        m for m in panel_markers
        if m in df_before.columns
        and m in df_after.columns
        and m in ctx.true_markers
        and pd.api.types.is_numeric_dtype(df_before[m])
        and pd.api.types.is_numeric_dtype(df_after[m])
    ]
    factors = [
        c for c in ["plate", "sample_name", "celltypes", "Time"]
        if c in df_before.columns and c in df_after.columns
    ]

    groups_before = sorted(df_before["group"].dropna().astype(str).unique()) if "group" in df_before else []
    groups_after = sorted(df_after["group"].dropna().astype(str).unique()) if "group" in df_after else []
    print(f"[ANALYZE] {panel_name}: markers={len(valid_markers)}, factors={factors}")
    print(f"[INFO] {panel_name} groups: before={groups_before}, after={groups_after}")

    before_result = variation_pca(
        df_before,
        markers=valid_markers,
        group_col="group",
        info_cols=factors,
        panel=panel_name,
    )
    after_result = variation_pca(
        df_after,
        markers=valid_markers,
        group_col="group",
        info_cols=factors,
        panel=panel_name,
    )
    return before_result, after_result


# Discover sibling split configs (for example D24 Panel1/Panel2). A split
# config declares only its own panel, so relying on cfg['panel'] alone
# incorrectly turns a two-panel study into a single-panel analysis.
_candidate_panels = ["Panel1", "Panel2"]
_inferred_paths = infer_panel_config_paths(
    os.path.abspath(study_config_path),
    _candidate_panels,
    current_panel=str(target_panel),
)
_panel_configs_resolved = all(
    _inferred_paths.get(p) and os.path.exists(_inferred_paths[p])
    for p in _candidate_panels
)

if _panel_configs_resolved:
    configured_panels = _candidate_panels
    panel_config_paths = {p: _inferred_paths[p] for p in configured_panels}
else:
    configured_panels = _configured_panels(study_config, target_panel)
    panel_config_paths = {p: os.path.abspath(study_config_path) for p in configured_panels}

panel_configs = {}
for _panel_name, _config_path in panel_config_paths.items():
    with open(_config_path) as _f:
        panel_configs[_panel_name] = json.load(_f)

print(f"[INFO] Panels discovered: {configured_panels}")
for _panel_name in configured_panels:
    print(f"  {_panel_name}: {panel_config_paths[_panel_name]}")

# Always recompute the current panel by biological group. df_b is already
# arcsinh-transformed raw data and df_a is the time-corrected matrix.
grouped_variance_results = {}
current_before, current_after = _grouped_panel_variance(
    df_b,
    df_a,
    str(target_panel),
    markers,
)
grouped_variance_results[str(target_panel)] = {
    "before": current_before,
    "after": current_after,
}

# Only load extra panels when the configuration explicitly declares them.
for panel_name in configured_panels:
    if panel_name == str(target_panel):
        continue

    panel_cfg = panel_configs[panel_name]
    with open(panel_cfg["outputs"]["markers_path"]) as _f:
        panel_marker_list = json.load(_f)

    _panel_sample_cap, _panel_source_count = resolve_matrix_sampling(
        panel_cfg["outputs"]["matrix_path"], target_dataset, panel_name
    )
    print(f"[LOAD] Reading {panel_name} for grouped cross-panel variance")
    df_panel_before = read_matrix_filtered_grouped(
        panel_cfg["outputs"]["matrix_path"],
        dataset=target_dataset,
        panel=panel_name,
        chunksize=matrix_chunksize,
        sample_per_group=_panel_sample_cap,
        group_cols=panel_sample_group_cols,
    )
    df_panel_after = read_matrix_filtered_grouped(
        panel_cfg["outputs"]["matrix_intermediate_path"],
        dataset=target_dataset,
        panel=panel_name,
        chunksize=matrix_chunksize,
        sample_per_group=_panel_sample_cap,
        group_cols=panel_sample_group_cols,
    )

    if df_panel_before.empty or df_panel_after.empty:
        print(f"[WARN] {panel_name} has no matching before/after cells; skipping it.")
        del df_panel_before, df_panel_after
        gc.collect()
        continue

    panel_marker_list = [m for m in panel_marker_list if m in df_panel_before.columns]
    df_panel_before = arcsinh(df_panel_before, panel_marker_list)
    panel_before, panel_after = _grouped_panel_variance(
        df_panel_before,
        df_panel_after,
        panel_name,
        panel_marker_list,
    )
    grouped_variance_results[panel_name] = {
        "before": panel_before,
        "after": panel_after,
    }
    del df_panel_before, df_panel_after
    gc.collect()


available_panels = [p for p in configured_panels if p in grouped_variance_results]
print(f"[INFO] Panels available for grouped variance plotting: {available_panels}")

if len(available_panels) == 1:
    panel_name = available_panels[0]
    # Keep grouped variance summaries in one study-level location even when
    # only one panel is currently available. A later two-panel run writes to
    # the same directory instead of scattering related figures by panel.
    variance_dir = os.path.join(ctx.output_dir, target_dataset, "Cross_Panel_Time_Variance")
    os.makedirs(variance_dir, exist_ok=True)
    save_path = os.path.join(
        variance_dir,
        f"grouped_variance_{target_dataset}_{panel_name}_Time.png",
    )
    variance_plot_df = plot_variance_bar(
        grouped_variance_results[panel_name]["before"],
        grouped_variance_results[panel_name]["after"],
        save_path=save_path,
    )
    print(f"[DONE] Group-stratified single-panel variance plot saved to: {save_path}")
else:
    p1_name, p2_name = available_panels[:2]
    variance_dir = os.path.join(ctx.output_dir, target_dataset, "Cross_Panel_Time_Variance")
    os.makedirs(variance_dir, exist_ok=True)
    save_path = os.path.join(
        variance_dir,
        f"grouped_variance_{target_dataset}_{p1_name}_vs_{p2_name}_Time.png",
    )
    variance_plot_df = plot_variance_bar(
        grouped_variance_results[p1_name]["before"],
        grouped_variance_results[p1_name]["after"],
        grouped_variance_results[p2_name]["before"],
        grouped_variance_results[p2_name]["after"],
        save_path=save_path,
    )
    print(f"[DONE] Group-stratified cross-panel variance plot saved to: {save_path}")

display(Image(filename=save_path))
gc.collect()
